# CITE-seq BMMC node-importance distributions

In [ ]:
mode = "all"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS14_S17_neurips2021_cite_bmmc_go_graph_distributions")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from matplotlib.colors import TwoSlopeNorm


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
if mode == "all":
    phenotypes = [
        "Lymph_prog",
        "CD4+_T_naive",
        "CD8+_T_naive",
        "Naive_CD20+_B_IGKC+",
        "Naive_CD20+_B_IGKC-",
        "Plasma_cell_IGKC+",
        "Plasma_cell_IGKC-",
        "Plasmablast_IGKC+",
        "Plasmablast_IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_labels = [
        "Lymph prog",
        "CD4+ T naive",
        "CD8+ T naive",
        "Naive CD20+ B IGKC+",
        "Naive CD20+ B IGKC-",
        "Plasma cell IGKC+",
        "Plasma cell IGKC-",
        "Plasmablast IGKC+",
        "Plasmablast IGKC-",
        "Erythroblast",
        "Normoblast",
        "Reticulocyte",
    ]
    phenotypes_dataset = phenotypes_labels


In [ ]:
(
    mdata,
    mdata_counts,
    adata,
    train_indices,
    val_indices,
    n_genes,
    n_regions,
    n_snps,
    labels_key,
    further_labels_keys,
    batch_key,
    patient_key,
    further_batch_keys,
    further_continuous_batch_keys,
    n_patient_covariates,
    protein_expression_obsm_key,
) = datasets.load_neurips2021_cite_BMMC(1.0, True, True)


In [ ]:
def aggregate_dataframes(graph_csv_list):
    num_rows = len(graph_csv_list[0])
    for df in graph_csv_list:
        if len(df) != num_rows:
            raise ValueError("Alle DataFrames müssen die gleiche Anzahl an Zeilen haben.")

    aggregated_roc = []
    aggregated_activations = []

    for row_idx in range(num_rows):

        roc_values = [df.iloc[row_idx]["predictors_label_valid_roc"] for df in graph_csv_list]

        mean_roc = np.mean(roc_values, axis=0)
        aggregated_roc.append(mean_roc)

        activations_arrays = [df.iloc[row_idx]["activations_valid"] for df in graph_csv_list]

        concatenated_activations = np.concatenate(activations_arrays, axis=1)

        mean_activations = np.mean(concatenated_activations, axis=1)
        aggregated_activations.append(mean_activations)

    aggregated_df = pd.DataFrame(
        index=graph_csv_list[0].index,
        data={
            "predictors_label_valid_roc": aggregated_roc,
            "activations_valid": aggregated_activations,
        },
    )

    return aggregated_df


In [ ]:
seeds = ["0"]

graph_csv_list = []
graph_csv_index_list = []
go_importances = []
for phenotype in tqdm(phenotypes):
    go_importances_phenotype = []
    for label in ["SparseVI"]:
        for modality in ["expression", "protein"]:
            if label == "SparseVI":
                graph_csv = pd.read_pickle(
                    f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
                )
            elif label == "MultiVI":
                graph_csv = pd.read_pickle(
                    f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
                )
            else:
                raise NotImplementedError("")
            go_importances_phenotype.append(np.array(graph_csv["predictors_label_valid_roc"]))
    go_importances.append(go_importances_phenotype)


In [ ]:
colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax2.grid(False)

for i, (sub_go_importances_phenotype, label) in enumerate(
    zip(go_importances_phenotype, ["SparseVI expression", "SparseVI protein"])
):
    if "expression" in label:
        ax1.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)
    else:
        ax2.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)

ax1.set_ylabel("prevalence (expression)")
ax2.set_ylabel("prevalence (protein)")
ax1.set_xlabel("GO Importance")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
set_figure_params('double', 3)
fig, axes = plt.subplots(3, 4)
axes = axes.flatten()
for i, (ax, go_importances_phenotype) in enumerate(zip(axes, go_importances)):
    ax2 = ax.twinx()
    ax2.grid(False)
    print(phenotypes_labels[i])
    ax.set_title(phenotypes_labels[i])
    for i, (sub_go_importances_phenotype, label) in enumerate(zip(go_importances_phenotype, ['SparseVI expression', 'SparseVI protein'])):
        if 'expression' in label:
            ax.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)
        else:
            ax2.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)
    ax.set_xlabel('GO importance')
    ax.set_ylabel('prevalence (expression)')
    ax2.set_ylabel('prevalence (protein)')
    ax.set_xlim(right=1.0)
handles = []
labels = []
for i, label in enumerate(['SparseVI expression', 'SparseVI protein']):
    handles.append(plt.Rectangle((0, 0), 1, 1, color=colors[i], alpha=0.5))
    labels.append(label)
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()
plt.subplots_adjust(wspace=0.7, hspace=1)
plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_cite_bmmc_go_importances_hist_SparseVI_modality_comparison.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_cite_bmmc_go_importances_hist_SparseVI_modality_comparison.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')


In [ ]:
colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

for i, (sub_go_importances_phenotype, label) in enumerate(
    zip(go_importances_phenotype, ["SparseVI expression", "MultiVI expression"])
):
    if "SparseVI" in label:
        ax1.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)
    else:
        ax2.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)

ax1.set_ylabel("prevalence (SparseVI)")
ax2.set_ylabel("prevalence (MultiVI)")
ax1.set_xlabel("GO Importance")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

for i, (sub_go_importances_phenotype, label) in enumerate(
    zip(
        go_importances_phenotype,
        ["SparseVI expression", "SparseVI protein", "MultiVI expression", "MultiVI protein"],
    )
):
    if "SparseVI" in label:
        ax1.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)
    else:
        ax2.hist(sub_go_importances_phenotype, bins=30, alpha=0.5, color=colors[i], label=label)

ax1.set_ylabel("Frequency (SparseVI)")
ax2.set_ylabel("Frequency (MultiVI)")
ax1.set_xlabel("GO Importance")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
seeds = ["0"]

graph_csv_list = []
graph_csv_index_list = []
go_importances = []
for modality in tqdm(["expression", "protein"]):
    go_importances_modality = []
    graph_csv_modality_list = []
    for (
        phenotype
    ) in phenotypes:
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_interaction_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        go_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_list.append(graph_csv_modality_list)
    graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
    go_importances.append(go_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_dense_list = []
graph_csv_dense_index_list = []
go_dense_importances = []
for modality in tqdm(["expression", "protein"]):
    go_dense_importances_modality = []
    graph_csv_dense_modality_list = []
    for (
        phenotype
    ) in phenotypes:
        graph_csv_dense_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_mmixTrue_no_gating_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_dense_s0/default/calc_go_terms_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_go_terms_evaluated_graph_csv.pkl"
        )
        graph_csv_dense_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_dense_phenotypes_list)
        graph_csv_dense_modality_list.append(graph_csv)
        go_dense_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_dense_list.append(graph_csv_dense_modality_list)
    graph_csv_dense_index_list.append(np.array(graph_csv_dense_modality_list[0].index))
    go_dense_importances.append(go_dense_importances_modality)


# GO importance statistics

In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params('double', 9)
    fig, axes = plt.subplots(9, 5)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, go_importances[modality_index])):
        print(phenotypes_labels[i])
        print(np.mean(go_importances[modality_index][i]))
        ax.hist(go_importances[modality_index][i], bins=50)
        ax.set_title(phenotypes_labels[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()


In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params('double', 9)
    fig, axes = plt.subplots(9, 5)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, go_dense_importances[modality_index])):
        print(phenotypes_labels[i])
        print(np.mean(go_dense_importances[modality_index][i]))
        ax.hist(go_dense_importances[modality_index][i], bins=50)
        ax.set_title(phenotypes_labels[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()


In [ ]:
lymph_prog = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Lymph prog"]
print(np.mean(lymph_prog[:, lymph_prog.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(lymph_prog[:, lymph_prog.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
lymph_prog


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Lymph prog"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
for phenotype in [
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "Lymph prog",
    "Erythroblast",
    "Normoblast",
]:
    print(phenotype)
    sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == phenotype]
    print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
    print(np.max(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
    print(np.min(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
    print(np.std(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Normoblast"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Erythroblast"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Naive CD20+ B IGKC+"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Naive CD20+ B IGKC-"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Normoblast"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata


In [ ]:
sub_adata = mdata.mod["prot"][mdata.mod["prot"].obs["cell_type"] == "Normoblast"]
print(np.mean(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense()))
plt.hist(sub_adata[:, sub_adata.var["gene_stable_id"] == "ENSG00000177455"].X.todense())
plt.show()
sub_adata
